In [1]:
import os
import glob
import math
import re

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FormatStrFormatter

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.9,
})

In [2]:
UNIT_TO_MILLION = {'K': 1e-3, 'M': 1.0, 'B': 1e3, 'T': 1e6}
PATTERN_FILENAME_RE = re.compile(r'^step[_-]?(\d+).*f1_macro_stat\.pt$', flags=re.I)

def parse_param_to_million(model_name: str):
    m = re.search(r'(?<!\d)(\d+(?:\.\d+)?)([KMBT])(?=_|$)', model_name)
    if not m:
        return None
    return float(m.group(1)) * UNIT_TO_MILLION[m.group(2)]

def parse_param_string_to_million(param_str):
    if param_str is None:
        return None
    s = str(param_str).strip().upper()
    m = re.match(r'^(\d+(?:\.\d+)?)([KMBT])$', s)
    if not m:
        return None
    return float(m.group(1)) * UNIT_TO_MILLION[m.group(2)]

def format_param_from_million(v_million: float):
    if v_million >= 1e6:
        return f'{v_million / 1e6:g}T'
    if v_million >= 1e3:
        return f'{v_million / 1e3:g}B'
    if v_million >= 1:
        return f'{v_million:g}M'
    return f'{v_million * 1e3:g}K'

def format_step_label(step):
    if step >= 1000 and step % 1000 == 0:
        return f'{step // 1000}k'
    if step >= 1000:
        return f'{step / 1000:g}k'
    return str(step)

def read_metric_from_dict(metrics):
    acc = None
    std = 0.0
    if isinstance(metrics, dict):
        if 'acc' in metrics:
            acc = float(metrics['acc'])
        elif 'test' in metrics:
            acc = float(metrics['test'])
        else:
            for v in metrics.values():
                try:
                    acc = float(v)
                    break
                except Exception:
                    continue
        if 'test_std' in metrics:
            std = float(metrics['test_std'])
        elif 'std' in metrics:
            std = float(metrics['std'])
        elif 'var' in metrics:
            std = math.sqrt(float(metrics['var']))
    else:
        try:
            acc = float(metrics)
        except Exception:
            acc = None
    return acc, std

def load_best_epoch_scores(folder):
    pt_files = sorted(glob.glob(os.path.join(folder, '*f1_macro_stat.pt')))
    epoch_data = {}
    for pt_file in pt_files:
        bn = os.path.basename(pt_file)
        if not PATTERN_FILENAME_RE.match(bn):
            continue
        try:
            data = torch.load(pt_file, map_location='cpu', weights_only=False)
            for ds, metrics in data.items():
                acc, std = read_metric_from_dict(metrics)
                if acc is not None:
                    epoch_data.setdefault(ds, []).append({'acc': acc, 'std': std})
        except Exception:
            continue
    dataset_accs = {}
    for ds, acc_list in epoch_data.items():
        if acc_list:
            dataset_accs[ds] = max(acc_list, key=lambda x: x['acc'])
    return dataset_accs

def source_blocks_from_keys(ds_keys):
    blocks = []
    if not ds_keys:
        return blocks
    start_idx = 0
    current_source = ds_keys[0][0]
    for idx, (source, _) in enumerate(ds_keys[1:], start=1):
        if source != current_source:
            blocks.append((current_source, start_idx, idx - 1))
            current_source = source
            start_idx = idx
    blocks.append((current_source, start_idx, len(ds_keys) - 1))
    return blocks

def build_xpos_fn(min_log, log_span, cluster_gap):
    def xpos(ds_idx, value, offset=0.0):
        base = ds_idx * (log_span + cluster_gap)
        slot = math.log10(value) - min_log
        return base + slot + offset
    return xpos

def set_unique_y_tick_formatter(ax, min_decimals=1, max_decimals=3):
    y_min, y_max = ax.get_ylim()
    ticks = [t for t in ax.get_yticks() if y_min <= t <= y_max]
    decimals = min_decimals
    for decimals in range(min_decimals, max_decimals + 1):
        labels = [f'{t:.{decimals}f}' for t in ticks]
        if len(labels) == len(set(labels)):
            break
    ax.yaxis.set_major_formatter(FormatStrFormatter(f'%.{decimals}f'))

def source_rank_by_performance(sums, counts):
    source_sums = {}
    source_counts = {}
    for k, total in sums.items():
        source = k[0]
        source_sums[source] = source_sums.get(source, 0.0) + total
        source_counts[source] = source_counts.get(source, 0) + counts.get(k, 0)
    source_names = list(source_sums.keys())
    source_names.sort(key=lambda source: (source_sums[source] / max(source_counts[source], 1), source))
    return {source: idx for idx, source in enumerate(source_names)}

def dataset_sort_key(k, sums, counts, source_rank):
    source, dataset = k
    avg_acc = sums[k] / max(counts[k], 1)
    return source_rank.get(source, len(source_rank)), avg_acc, dataset

def dataset_keys_for_task(rows):
    sums = {}
    counts = {}
    for p in rows:
        k = (p['source'], p['dataset'])
        sums[k] = sums.get(k, 0.0) + float(p['acc'])
        counts[k] = counts.get(k, 0) + 1
    keys = list(sums.keys())
    source_rank = source_rank_by_performance(sums, counts)
    keys.sort(key=lambda k: dataset_sort_key(k, sums, counts, source_rank))
    return keys

def dataset_keys_for_task_with_baselines(series, baseline_stats):
    sums = {}
    counts = {}
    for walk in series:
        for p in series[walk]:
            k = (p['source'], p['dataset'])
            sums[k] = sums.get(k, 0.0) + float(p['acc'])
            counts[k] = counts.get(k, 0) + 1
    for b in baseline_stats.values():
        for r in b['rows']:
            k = (r['source'], r['dataset'])
            sums[k] = sums.get(k, 0.0) + float(r['acc'])
            counts[k] = counts.get(k, 0) + 1
    keys = list(sums.keys())
    source_rank = source_rank_by_performance(sums, counts)
    keys.sort(key=lambda k: dataset_sort_key(k, sums, counts, source_rank))
    return keys

# ---------- Plot 001 (walk length vs model size, with baselines) ----------
WALK_ORDER = ('8', '16', '32', '64')
MODEL_FAMILY_PREFIX = 'corpus_full'
MODEL_SIZE_SPECS = ['3M', '15M', '36M', '317M']
MODEL_SUFFIX_TEMPLATE = '{prefix}_{size}_05_19_wl_{walk}_LP_model'
BASELINE_KEY_TEMPLATE = '{prefix}'
BASELINE_FOLDER_TEMPLATE = '{prefix}_LP_model{suffix}'
BASELINE_PREFIX_TO_PARAM = {
    'scgpt_spatial_emb': '50.1M',
    'nicheformer_emb': '49.3M',
    'novae_emb': '33.6M',
}
BASELINE_PREFIX_TO_LABEL = {
    'scgpt_spatial_emb': 'scGPT-spatial',
    'nicheformer_emb': 'Nicheformer',
    'novae_emb': 'Novae',
}
BASELINE_PREFIX_TO_COLOR = {
    'scgpt_spatial_emb': '#C9B6E4',
    'nicheformer_emb': '#7DC69B',
    'novae_emb': '#9BD7F3',
}
BASELINE_MODEL_PARAMS = {
    BASELINE_KEY_TEMPLATE.format(prefix=prefix): param
    for prefix, param in BASELINE_PREFIX_TO_PARAM.items()
}
BASELINE_MARKERS = ['D', 's', '^', 'P', 'X', 'v', '<', '>', '*', 'h']
GRADIENT_BASE_COLOR = '#F2A1A7'
ORDERED_ALPHAS = [0.25, 0.50, 0.75, 1.00]
ORDERED_COLORS = [mcolors.to_rgba(GRADIENT_BASE_COLOR, alpha=a) for a in ORDERED_ALPHAS]
SOURCE_BAND_COLORS = ['#E7EEF8', '#F1E4DA', '#E5F0EA', '#F3E2E7', '#EEF3D8']
DOT_SIZE = 100
BAND_ALPHA = 0.5
LEGEND_BAND_ALPHA = 0.85
CLUSTER_GAP = 2.0
AMPLIFIER_MIN = 0.85
AMPLIFIER_MAX = 1.35

def model_size_alpha(param_million):
    size_params = [parse_param_string_to_million(size) for size in MODEL_SIZE_SPECS]
    size_params = [p for p in size_params if p is not None]
    if len(size_params) <= 1:
        return ORDERED_ALPHAS[-1]
    log_min = math.log10(min(size_params))
    log_max = math.log10(max(size_params))
    t = (math.log10(param_million) - log_min) / max(log_max - log_min, 1e-6)
    t = min(max(t, 0.0), 1.0)
    return ORDERED_ALPHAS[0] + t * (ORDERED_ALPHAS[-1] - ORDERED_ALPHAS[0])

def walk_amplifier(walk):
    if len(WALK_ORDER) == 1:
        return 1.0
    walk_vals = [int(w) for w in WALK_ORDER]
    walk_min = min(walk_vals)
    walk_max = max(walk_vals)
    walk_val = int(walk)
    t = (walk_val - walk_min) / max(walk_max - walk_min, 1e-6)
    return AMPLIFIER_MIN + t * (AMPLIFIER_MAX - AMPLIFIER_MIN)

def model_size_amplifier(param_million):
    size_params = [parse_param_string_to_million(size) for size in MODEL_SIZE_SPECS]
    size_params = [p for p in size_params if p is not None]
    if len(size_params) <= 1:
        return 1.0
    log_min = math.log10(min(size_params))
    log_max = math.log10(max(size_params))
    t = (math.log10(param_million) - log_min) / max(log_max - log_min, 1e-6)
    t = min(max(t, 0.0), 1.0)
    return AMPLIFIER_MIN + t * (AMPLIFIER_MAX - AMPLIFIER_MIN)

def build_family_model_key(size: str, walk: str):
    return MODEL_SUFFIX_TEMPLATE.format(prefix=MODEL_FAMILY_PREFIX, size=size, walk=walk)

def build_baseline_folder_name(prefix: str, suffix: str):
    return BASELINE_FOLDER_TEMPLATE.format(prefix=prefix, suffix=suffix)

def model_folder_map_001(base_dir, spaGFM_dir, suffix):
    model_folders = {
        BASELINE_KEY_TEMPLATE.format(prefix=prefix): os.path.join(base_dir, build_baseline_folder_name(prefix, suffix))
        for prefix in BASELINE_PREFIX_TO_PARAM
    }
    for walk in WALK_ORDER:
        for size in MODEL_SIZE_SPECS:
            key = build_family_model_key(size, walk)
            model_folders[key] = os.path.join(spaGFM_dir, f'{key}{suffix}')
    return model_folders

def load_full_results_001(task_cfg, resources):
    suffix = task_cfg['suffix']
    full_results = {}
    model_keys = set()
    for walk in WALK_ORDER:
        for size in MODEL_SIZE_SPECS:
            model_keys.add(build_family_model_key(size, walk))
    model_keys.update(BASELINE_MODEL_PARAMS.keys())
    for resource in resources:
        base_dir = resource['base_dir']
        spaGFM_dir = resource['spaGFM_dir']
        source_name = resource['name']
        folders = model_folder_map_001(base_dir, spaGFM_dir, suffix)
        full_results.setdefault(source_name, {})
        for model in sorted(model_keys):
            folder = folders.get(model)
            if not folder or not os.path.isdir(folder):
                continue
            dataset_accs = {}
            if model in BASELINE_MODEL_PARAMS:
                baseline_files = sorted(glob.glob(os.path.join(folder, '*f1_macro_stat.pt')))
                baseline_files = [f for f in baseline_files if not os.path.basename(f).lower().startswith('epoch')]
                if not baseline_files:
                    continue
                for pt_file in baseline_files:
                    try:
                        data = torch.load(pt_file, map_location='cpu', weights_only=False)
                        if isinstance(data, dict) and any(k in data for k in ('acc', 'test', 'val', 'train')):
                            ds = os.path.splitext(os.path.basename(pt_file))[0]
                            if '_linear_probe_' in ds:
                                ds = ds.split('_linear_probe_')[0]
                            acc, std = read_metric_from_dict(data)
                            if acc is not None:
                                dataset_accs[ds] = {'acc': acc, 'std': std}
                        elif isinstance(data, dict):
                            for ds, metrics in data.items():
                                acc, std = read_metric_from_dict(metrics)
                                if acc is not None:
                                    dataset_accs[ds] = {'acc': acc, 'std': std}
                    except Exception:
                        continue
            else:
                dataset_accs = load_best_epoch_scores(folder)
            for ds, score_dict in dataset_accs.items():
                full_results[source_name].setdefault(ds, {})[model] = score_dict
    return full_results

def build_plot_data_001(full_results):
    series = {walk: [] for walk in WALK_ORDER}
    baseline_stats = {}
    for source_name, datasets in full_results.items():
        for ds_name, score_map in datasets.items():
            for model_name, metrics in score_map.items():
                if not isinstance(metrics, dict) or ('acc' not in metrics):
                    continue
                if model_name in BASELINE_MODEL_PARAMS:
                    param_million = parse_param_string_to_million(BASELINE_MODEL_PARAMS.get(model_name))
                    if param_million is None:
                        continue
                    baseline_stats.setdefault(model_name, {'param_million': float(param_million), 'rows': []})
                    baseline_stats[model_name]['rows'].append({
                        'source': source_name,
                        'dataset': ds_name,
                        'acc': float(metrics['acc']),
                        'std': float(metrics['std']) if metrics.get('std') is not None else 0.0,
                    })
                    continue
                param_million = parse_param_to_million(model_name)
                if param_million is None:
                    continue
                walk_match = re.search(r'_wl_(8|16|32|64)_', model_name)
                if not walk_match:
                    continue
                walk = walk_match.group(1)
                if walk not in series:
                    continue
                series[walk].append({
                    'source': source_name,
                    'dataset': ds_name,
                    'model': model_name,
                    'param_million': float(param_million),
                    'acc': float(metrics['acc']),
                    'std': float(metrics['std']) if metrics.get('std') is not None else 0.0,
                })
    return series, baseline_stats

def draw_task_dataset_axis_001(ax, task_title, series, baseline_stats, source_band_colors, show_legend=False):
    ds_keys = dataset_keys_for_task_with_baselines(series, baseline_stats)
    if not ds_keys:
        ax.set_title(task_title, fontsize=20.0, pad=6)
        return None
    spaGFM_params = sorted({p['param_million'] for w in WALK_ORDER for p in series[w]})
    all_params = sorted(set(spaGFM_params) | {b['param_million'] for b in baseline_stats.values()})
    if not all_params:
        ax.set_title(task_title, fontsize=20.0, pad=6)
        return None
    log_values = [math.log10(p) for p in all_params]
    min_log = min(log_values)
    max_log = max(log_values)
    log_span = max(max_log - min_log, 1e-6)
    xpos = build_xpos_fn(min_log, log_span, CLUSTER_GAP)
    source_blocks = source_blocks_from_keys(ds_keys)
    for source, start_idx, end_idx in source_blocks:
        base_left = start_idx * (log_span + CLUSTER_GAP) - (CLUSTER_GAP / 2.0)
        base_right = end_idx * (log_span + CLUSTER_GAP) + log_span + (CLUSTER_GAP / 2.0)
        band_color = source_band_colors.get(source, '#F2F2F2')
        ax.axvspan(base_left, base_right, color=band_color, alpha=BAND_ALPHA, zorder=0)
    for walk in WALK_ORDER:
        pts = series[walk]
        ds_grouped = {}
        for p in pts:
            ds_grouped.setdefault((p['source'], p['dataset']), []).append(p)
        for ds_idx, ds_key in enumerate(ds_keys):
            ds_pts = ds_grouped.get(ds_key, [])
            if not ds_pts:
                continue
            ds_pts = sorted(ds_pts, key=lambda d: d['param_million'])
            offset = {'8': -0.06, '16': 0.06, '32': 0.12, '64': 0.18}[walk]
            xs = [xpos(ds_idx, p['param_million'], offset=offset) for p in ds_pts]
            ys = [p['acc'] for p in ds_pts]
            if len(xs) > 1:
                ax.plot(xs, ys, color=mcolors.to_rgba(GRADIENT_BASE_COLOR, alpha=0.28), linewidth=1.2, zorder=2)
            point_color = mcolors.to_rgba(GRADIENT_BASE_COLOR, alpha=ORDERED_ALPHAS[WALK_ORDER.index(walk)])
            sizes = [DOT_SIZE * model_size_amplifier(p['param_million']) for p in ds_pts]
            ax.scatter(xs, ys, s=sizes, marker='o', color=point_color, edgecolor='white', linewidth=0.8, zorder=4)
    baseline_names = sorted(baseline_stats.keys())
    baseline_marker_map = {
        b: BASELINE_MARKERS[i % len(BASELINE_MARKERS)] for i, b in enumerate(baseline_names)
    }
    for b_i, b in enumerate(baseline_names):
        v = baseline_stats[b]
        param_million = v['param_million']
        rows_by_ds = {}
        for row in v['rows']:
            rows_by_ds.setdefault((row['source'], row['dataset']), []).append(row)
        jitter = (b_i - (len(baseline_names) - 1) / 2.0) * 0.05
        for ds_idx, ds_key in enumerate(ds_keys):
            rows = rows_by_ds.get(ds_key, [])
            if not rows:
                continue
            x = xpos(ds_idx, param_million) + jitter
            ys = [r['acc'] for r in rows]
            ax.scatter([x] * len(ys), ys, s=DOT_SIZE * 0.9, marker=baseline_marker_map[b], color=BASELINE_PREFIX_TO_COLOR.get(b, '#4A4A4A'), edgecolor=BASELINE_PREFIX_TO_COLOR.get(b, '#4A4A4A'), linewidth=0.8, alpha=1.0, zorder=3)
    y_min, y_max = ax.get_ylim()
    y_pad = 0.04 * (y_max - y_min if y_max > y_min else 1.0)
    ax.set_ylim(y_min - y_pad, y_max + y_pad)
    center_idx = len(ds_keys) // 2
    base0 = center_idx * (log_span + CLUSTER_GAP)
    x_ticks = [base0 + (math.log10(p) - min_log) for p in spaGFM_params]
    x_labels = [format_param_from_million(p) for p in spaGFM_params]
    x_left = -0.4
    x_right = (len(ds_keys) - 1) * (log_span + CLUSTER_GAP) + log_span + 0.4
    ax.set_xlim(x_left - 0.4, x_right + 0.4)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([''] * len(x_ticks))
    ax.tick_params(axis='x', length=6, width=0.9, pad=6)
    label_xs = np.linspace(x_left + 0.6, x_right - 0.6, len(x_labels))
    axis_y = 0.0
    label_y = -0.155
    for tick_x, label_x, label in zip(x_ticks, label_xs, x_labels):
        ax.annotate(label, xy=(tick_x, axis_y), xycoords=ax.get_xaxis_transform(), xytext=(label_x, label_y), textcoords=ax.get_xaxis_transform(), ha='center', va='top', fontsize=12.0, fontweight='regular', arrowprops=dict(arrowstyle='-', color='#8C8C8C', lw=0.45, alpha=0.42), clip_on=False)
    for i in range(len(ds_keys) - 1):
        boundary = (i + 1) * (log_span + CLUSTER_GAP) - (CLUSTER_GAP / 2.0)
        ax.axvline(boundary, color='#D0D0D0', linestyle='-', linewidth=0.55, alpha=0.24, zorder=1)
    ax.set_title(task_title, fontsize=20.0, pad=6)
    ax.tick_params(axis='y', labelsize=12.0)
    set_unique_y_tick_formatter(ax)
    ax.grid(True, axis='y', alpha=0.16, linewidth=0.65, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if not show_legend:
        return None
    walk_handles = [
        Line2D([0], [0], color=mcolors.to_rgba(GRADIENT_BASE_COLOR, alpha=ORDERED_ALPHAS[WALK_ORDER.index(w)]), linestyle='None',
               marker='o', markersize=6.8,
               markerfacecolor=mcolors.to_rgba(GRADIENT_BASE_COLOR, alpha=ORDERED_ALPHAS[WALK_ORDER.index(w)]),
               markeredgecolor='white', markeredgewidth=0.8, label=w)
        for w in WALK_ORDER
    ]
    model_size_handles = [
        Line2D([0], [0], color=mcolors.to_rgba(GRADIENT_BASE_COLOR, alpha=0.72), linestyle='None',
               marker='o', markersize=6.8 * model_size_amplifier(parse_param_string_to_million(size)),
               markerfacecolor=mcolors.to_rgba(GRADIENT_BASE_COLOR, alpha=0.72),
               markeredgecolor='white', markeredgewidth=0.8, label=size)
        for size in MODEL_SIZE_SPECS
    ]
    source_names = [source for source, _, _ in source_blocks]
    source_names += [name for name in source_band_colors if name not in source_names]
    source_handles = [
        Patch(facecolor=source_band_colors[name], edgecolor='#B8B8B8', linewidth=0.45, label=name, alpha=LEGEND_BAND_ALPHA)
        for name in source_names
    ]
    base_handles = [
        Line2D([0], [0], color=BASELINE_PREFIX_TO_COLOR.get(k, '#4A4A4A'), linestyle='None',
               marker=BASELINE_MARKERS[i % len(BASELINE_MARKERS)],
               markersize=6.8, markerfacecolor=BASELINE_PREFIX_TO_COLOR.get(k, '#4A4A4A'), markeredgecolor=BASELINE_PREFIX_TO_COLOR.get(k, '#4A4A4A'), markeredgewidth=0.8,
               label=BASELINE_PREFIX_TO_LABEL.get(k, k.replace('_emb', '').replace('_', ' ').title()))
        for i, k in enumerate(sorted(BASELINE_MODEL_PARAMS.keys()))
    ]
    return walk_handles, model_size_handles, source_handles, base_handles

# ---------- Plot 002 (training steps) ----------
STEPS = ['15000', '45000', '75000', '150000']
MODEL_FAMILY_PREFIX_2 = 'corpus_full'
MODEL_SIZE_SPECS_2 = ['317M']
WALK_LENGTHS_2 = ['8']
MODEL_SUFFIX_TEMPLATE_2 = '{prefix}_{size}_05_19_wl_{walk}_n_step_{step}_LP_model'

def build_family_model_key_2(size: str, walk: str, step: str):
    return MODEL_SUFFIX_TEMPLATE_2.format(prefix=MODEL_FAMILY_PREFIX_2, size=size, walk=walk, step=step)

def model_folder_map_2(spaGFM_dir, suffix):
    model_folders = {}
    for step in STEPS:
        for size in MODEL_SIZE_SPECS_2:
            for walk in WALK_LENGTHS_2:
                key = build_family_model_key_2(size, walk, step)
                model_folders[key] = os.path.join(spaGFM_dir, f'{key}{suffix}')
    return model_folders

def load_full_results_2(task_cfg, resources):
    suffix = task_cfg['suffix']
    full_results = {}
    model_keys = set()
    for step in STEPS:
        for size in MODEL_SIZE_SPECS_2:
            for walk in WALK_LENGTHS_2:
                model_keys.add(build_family_model_key_2(size, walk, step))
    for resource in resources:
        spaGFM_dir = resource['spaGFM_dir']
        source_name = resource['name']
        folders = model_folder_map_2(spaGFM_dir, suffix)
        full_results.setdefault(source_name, {})
        for model in sorted(model_keys):
            folder = folders.get(model)
            if not folder or not os.path.isdir(folder):
                continue
            dataset_accs = load_best_epoch_scores(folder)
            for ds, score_dict in dataset_accs.items():
                full_results[source_name].setdefault(ds, {})[model] = score_dict
    return full_results

def build_plot_data_2(full_results):
    rows = []
    step_order = tuple(int(s) for s in STEPS)
    for source_name, datasets in full_results.items():
        for ds_name, score_map in datasets.items():
            for model_name, metrics in score_map.items():
                if not isinstance(metrics, dict) or ('acc' not in metrics):
                    continue
                step_match = re.search(r'_step_(\d+)', model_name)
                if not step_match:
                    continue
                step = int(step_match.group(1))
                if step not in step_order:
                    continue
                rows.append({
                    'source': source_name,
                    'dataset': ds_name,
                    'model': model_name,
                    'step': step,
                    'acc': float(metrics['acc']),
                    'std': float(metrics['std']) if metrics.get('std') is not None else 0.0,
                })
    return rows

def step_amplifier(step, step_order):
    if len(step_order) == 1:
        return 1.0
    step_min = min(step_order)
    step_max = max(step_order)
    t = (step - step_min) / max(step_max - step_min, 1e-6)
    return AMPLIFIER_MIN + t * (AMPLIFIER_MAX - AMPLIFIER_MIN)

def draw_task_dataset_axis_2(ax, task_title, rows, source_band_colors):
    step_order = tuple(int(s) for s in STEPS)
    step_color_map = {step: ORDERED_COLORS[i % len(ORDERED_COLORS)] for i, step in enumerate(step_order)}
    ds_keys = dataset_keys_for_task(rows)
    if not ds_keys:
        ax.set_title(task_title, fontsize=20.0, pad=6)
        return
    log_values = [math.log10(step) for step in step_order]
    min_log = min(log_values)
    max_log = max(log_values)
    log_span = max(max_log - min_log, 1e-6)
    xpos = build_xpos_fn(min_log, log_span, CLUSTER_GAP)
    source_blocks = source_blocks_from_keys(ds_keys)
    for source, start_idx, end_idx in source_blocks:
        base_left = start_idx * (log_span + CLUSTER_GAP) - (CLUSTER_GAP / 2.0)
        base_right = end_idx * (log_span + CLUSTER_GAP) + log_span + (CLUSTER_GAP / 2.0)
        band_color = source_band_colors.get(source, '#F2F2F2')
        ax.axvspan(base_left, base_right, color=band_color, alpha=BAND_ALPHA, zorder=0)
    rows_by_ds = {}
    for p in rows:
        rows_by_ds.setdefault((p['source'], p['dataset']), []).append(p)
    for ds_idx, ds_key in enumerate(ds_keys):
        ds_pts = rows_by_ds.get(ds_key, [])
        if not ds_pts:
            continue
        ds_pts = sorted(ds_pts, key=lambda d: d['step'])
        xs = [xpos(ds_idx, p['step']) for p in ds_pts]
        ys = [p['acc'] for p in ds_pts]
        if len(xs) > 1:
            ax.plot(xs, ys, color='#6F6F6F', linewidth=1.1, alpha=0.45, zorder=2)
        for p in ds_pts:
            amp = step_amplifier(p['step'], step_order)
            ax.scatter(xpos(ds_idx, p['step']), p['acc'], s=DOT_SIZE * amp, color=step_color_map[p['step']], edgecolor='white', linewidth=0.8, zorder=4)
    y_min, y_max = ax.get_ylim()
    y_pad = 0.04 * (y_max - y_min if y_max > y_min else 1.0)
    ax.set_ylim(y_min - y_pad, y_max + y_pad)
    center_idx = len(ds_keys) // 2
    base0 = center_idx * (log_span + CLUSTER_GAP)
    x_ticks = [base0 + (math.log10(step) - min_log) for step in step_order]
    x_labels = [format_step_label(step) for step in step_order]
    x_left = -0.4
    x_right = (len(ds_keys) - 1) * (log_span + CLUSTER_GAP) + log_span + 0.4
    ax.set_xlim(x_left - 0.4, x_right + 0.4)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([''] * len(x_ticks))
    ax.tick_params(axis='x', length=6, width=0.9, pad=6)
    label_xs = np.linspace(x_left + 0.6, x_right - 0.6, len(x_labels))
    axis_y = 0.0
    label_y = -0.155
    for tick_x, label_x, label in zip(x_ticks, label_xs, x_labels):
        ax.annotate(label, xy=(tick_x, axis_y), xycoords=ax.get_xaxis_transform(), xytext=(label_x, label_y), textcoords=ax.get_xaxis_transform(), ha='center', va='top', fontsize=12.0, fontweight='regular', arrowprops=dict(arrowstyle='-', color='#8C8C8C', lw=0.45, alpha=0.42), clip_on=False)
    for i in range(len(ds_keys) - 1):
        boundary = (i + 1) * (log_span + CLUSTER_GAP) - (CLUSTER_GAP / 2.0)
        ax.axvline(boundary, color='#D0D0D0', linestyle='-', linewidth=0.55, alpha=0.24, zorder=1)
    ax.set_title(task_title, fontsize=20.0, pad=6)
    ax.tick_params(axis='y', labelsize=12.0)
    set_unique_y_tick_formatter(ax)
    ax.grid(True, axis='y', alpha=0.16, linewidth=0.65, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# ---------- Plot 003 (n-walks) ----------
N_WALKS = ['16', '32', '64', '128']
MODEL_FAMILY_PREFIX_3 = 'corpus_full'
MODEL_SIZE_SPECS_3 = ['317M']
WALK_LENGTHS_3 = ['8']
MODEL_SUFFIX_TEMPLATE_3 = '{prefix}_{size}_05_19_wl_{walk}_n_walk_{n_walk}_LP_model'

def build_family_model_key_3(size: str, walk: str, n_walk: str):
    return MODEL_SUFFIX_TEMPLATE_3.format(prefix=MODEL_FAMILY_PREFIX_3, size=size, walk=walk, n_walk=n_walk)

def model_folder_map_3(spaGFM_dir, suffix):
    model_folders = {}
    for n_walk in N_WALKS:
        for size in MODEL_SIZE_SPECS_3:
            for walk in WALK_LENGTHS_3:
                key = build_family_model_key_3(size, walk, n_walk)
                model_folders[key] = os.path.join(spaGFM_dir, f'{key}{suffix}')
    return model_folders

def load_full_results_3(task_cfg, resources):
    suffix = task_cfg['suffix']
    full_results = {}
    model_keys = set()
    for n_walk in N_WALKS:
        for size in MODEL_SIZE_SPECS_3:
            for walk in WALK_LENGTHS_3:
                model_keys.add(build_family_model_key_3(size, walk, n_walk))
    for resource in resources:
        spaGFM_dir = resource['spaGFM_dir']
        source_name = resource['name']
        folders = model_folder_map_3(spaGFM_dir, suffix)
        full_results.setdefault(source_name, {})
        for model in sorted(model_keys):
            folder = folders.get(model)
            if not folder or not os.path.isdir(folder):
                continue
            dataset_accs = load_best_epoch_scores(folder)
            for ds, score_dict in dataset_accs.items():
                full_results[source_name].setdefault(ds, {})[model] = score_dict
    return full_results

def build_plot_data_3(full_results):
    rows = []
    n_walk_order = tuple(int(n) for n in N_WALKS)
    for source_name, datasets in full_results.items():
        for ds_name, score_map in datasets.items():
            for model_name, metrics in score_map.items():
                if not isinstance(metrics, dict) or ('acc' not in metrics):
                    continue
                n_walk_match = re.search(r'_n_walk_(\d+)', model_name)
                if not n_walk_match:
                    continue
                n_walk = int(n_walk_match.group(1))
                if n_walk not in n_walk_order:
                    continue
                rows.append({
                    'source': source_name,
                    'dataset': ds_name,
                    'model': model_name,
                    'n_walk': n_walk,
                    'acc': float(metrics['acc']),
                    'std': float(metrics['std']) if metrics.get('std') is not None else 0.0,
                })
    return rows

def draw_task_dataset_axis_3(ax, task_title, rows, source_band_colors):
    n_walk_order = tuple(int(n) for n in N_WALKS)
    colors = ORDERED_COLORS
    n_walk_color_map = {n_walk: colors[i % len(colors)] for i, n_walk in enumerate(n_walk_order)}
    ds_keys = dataset_keys_for_task(rows)
    if not ds_keys:
        ax.set_title(task_title, fontsize=20.0, pad=6)
        return
    log_values = [math.log10(n_walk) for n_walk in n_walk_order]
    min_log = min(log_values)
    max_log = max(log_values)
    log_span = max(max_log - min_log, 1e-6)
    xpos = build_xpos_fn(min_log, log_span, CLUSTER_GAP)
    source_blocks = source_blocks_from_keys(ds_keys)
    for source, start_idx, end_idx in source_blocks:
        base_left = start_idx * (log_span + CLUSTER_GAP) - (CLUSTER_GAP / 2.0)
        base_right = end_idx * (log_span + CLUSTER_GAP) + log_span + (CLUSTER_GAP / 2.0)
        band_color = source_band_colors.get(source, '#F2F2F2')
        ax.axvspan(base_left, base_right, color=band_color, alpha=BAND_ALPHA, zorder=0)
    rows_by_ds = {}
    for p in rows:
        rows_by_ds.setdefault((p['source'], p['dataset']), []).append(p)
    for ds_idx, ds_key in enumerate(ds_keys):
        ds_pts = rows_by_ds.get(ds_key, [])
        if not ds_pts:
            continue
        ds_pts = sorted(ds_pts, key=lambda d: d['n_walk'])
        xs = [xpos(ds_idx, p['n_walk']) for p in ds_pts]
        ys = [p['acc'] for p in ds_pts]
        if len(xs) > 1:
            ax.plot(xs, ys, color='#6F6F6F', linewidth=1.1, alpha=0.45, zorder=2)
        for p in ds_pts:
            t = (p['n_walk'] - min(n_walk_order)) / max(max(n_walk_order) - min(n_walk_order), 1e-6)
            amp = AMPLIFIER_MIN + t * (AMPLIFIER_MAX - AMPLIFIER_MIN)
            ax.scatter(xpos(ds_idx, p['n_walk']), p['acc'], s=DOT_SIZE * amp, color=n_walk_color_map[p['n_walk']], edgecolor='white', linewidth=0.8, zorder=4)
    y_min, y_max = ax.get_ylim()
    y_pad = 0.04 * (y_max - y_min if y_max > y_min else 1.0)
    ax.set_ylim(y_min - y_pad, y_max + y_pad)
    center_idx = len(ds_keys) // 2
    base0 = center_idx * (log_span + CLUSTER_GAP)
    x_ticks = [base0 + (math.log10(n_walk) - min_log) for n_walk in n_walk_order]
    x_labels = [format_step_label(n_walk) for n_walk in n_walk_order]
    x_left = -0.4
    x_right = (len(ds_keys) - 1) * (log_span + CLUSTER_GAP) + log_span + 0.4
    ax.set_xlim(x_left - 0.4, x_right + 0.4)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([''] * len(x_ticks))
    ax.tick_params(axis='x', length=6, width=0.9, pad=6)
    label_xs = np.linspace(x_left + 0.6, x_right - 0.6, len(x_labels))
    axis_y = 0.0
    label_y = -0.155
    for tick_x, label_x, label in zip(x_ticks, label_xs, x_labels):
        ax.annotate(label, xy=(tick_x, axis_y), xycoords=ax.get_xaxis_transform(), xytext=(label_x, label_y), textcoords=ax.get_xaxis_transform(), ha='center', va='top', fontsize=12.0, fontweight='regular', arrowprops=dict(arrowstyle='-', color='#8C8C8C', lw=0.45, alpha=0.42), clip_on=False)
    for i in range(len(ds_keys) - 1):
        boundary = (i + 1) * (log_span + CLUSTER_GAP) - (CLUSTER_GAP / 2.0)
        ax.axvline(boundary, color='#D0D0D0', linestyle='-', linewidth=0.55, alpha=0.24, zorder=1)
    ax.set_title(task_title, fontsize=20.0, pad=6)
    ax.tick_params(axis='y', labelsize=12.0)
    set_unique_y_tick_formatter(ax)
    ax.grid(True, axis='y', alpha=0.16, linewidth=0.65, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# ---------- Build combined figure ----------
RESOURCES_001 = [
    {'name': 'Human Lung', 'base_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Lung_benchmark/baseline_dir/05_19_26',
     'spaGFM_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Lung_benchmark/g2pm_dir/05_19_26'},
    {'name': 'Human Liver', 'base_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Liver_benchmark/baseline_dir/05_19_26',
     'spaGFM_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Liver_benchmark/g2pm_dir/05_19_26'},
    {'name': 'Human Cortex', 'base_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Cortex_benchmark/baseline_dir/05_19_26',
     'spaGFM_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Cortex_benchmark/g2pm_dir/05_19_26'}
 ]
RESOURCES_002 = [
    {'name': 'Human Lung', 'base_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Lung_benchmark/baseline_dir/05_19_26',
     'spaGFM_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Lung_benchmark/g2pm_dir/05_19_26'},
    {'name': 'Human Liver', 'base_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Liver_benchmark/baseline_dir/05_19_26',
     'spaGFM_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Liver_benchmark/g2pm_dir/05_19_26'},
    {'name': 'Human Cortex', 'base_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Cortex_benchmark/baseline_dir/05_19_26',
     'spaGFM_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Cortex_benchmark/g2pm_dir/05_19_26'}
 ]
RESOURCES_003 = [
    {'name': 'Human Lung', 'base_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Lung_benchmark/baseline_dir/05_19_26',
     'spaGFM_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Lung_benchmark/g2pm_dir/05_19_26'},
    {'name': 'Human Liver', 'base_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Liver_benchmark/baseline_dir/05_19_26',
     'spaGFM_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Liver_benchmark/g2pm_dir/05_19_26'},
    {'name': 'Human Cortex', 'base_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Cortex_benchmark/baseline_dir/05_19_26',
     'spaGFM_dir': '/fs/ess/PAS1475/yzhong/sf_project/scripts/CosMx_Cortex_benchmark/g2pm_dir/05_19_26'}
 ]
TASKS = {
    'Niche task': {'suffix': '_niche', 'title': 'Niche task'},
    'Cell type task': {'suffix': '_ct', 'title': 'Cell type task'},
}

fig, axes = plt.subplots(3, 2, figsize=(13.0, 16.2), sharey=False, constrained_layout=False, dpi=600)
fig.patch.set_facecolor('white')
for row in axes:
    for ax in row:
        ax.set_facecolor('white')
        ax.set_box_aspect(1.05)
    row[0].set_anchor('E')
    row[1].set_anchor('W')

# Row 1 (001)
task_data_001 = {}
for task_name, cfg in TASKS.items():
    fr = load_full_results_001(cfg, RESOURCES_001)
    series, baseline_stats = build_plot_data_001(fr)
    task_data_001[task_name] = {'series': series, 'baseline_stats': baseline_stats, 'title': cfg['title']}
source_colors_001 = {r['name']: SOURCE_BAND_COLORS[i % len(SOURCE_BAND_COLORS)] for i, r in enumerate(RESOURCES_001)}
legend_bits = draw_task_dataset_axis_001(axes[0, 0], task_data_001['Niche task']['title'], task_data_001['Niche task']['series'], task_data_001['Niche task']['baseline_stats'], source_colors_001, show_legend=True)
draw_task_dataset_axis_001(axes[0, 1], task_data_001['Cell type task']['title'], task_data_001['Cell type task']['series'], task_data_001['Cell type task']['baseline_stats'], source_colors_001, show_legend=False)
axes[0, 0].set_ylabel('F1-macro', fontsize=15.0)
axes[0, 1].set_ylabel('')

# Row 2 (003)
task_data_003 = {}
for task_name, cfg in TASKS.items():
    fr = load_full_results_3(cfg, RESOURCES_003)
    rows = build_plot_data_3(fr)
    task_data_003[task_name] = {'rows': rows, 'title': cfg['title']}
source_colors_003 = {r['name']: SOURCE_BAND_COLORS[i % len(SOURCE_BAND_COLORS)] for i, r in enumerate(RESOURCES_003)}
draw_task_dataset_axis_3(axes[1, 0], task_data_003['Niche task']['title'], task_data_003['Niche task']['rows'], source_colors_003)
draw_task_dataset_axis_3(axes[1, 1], task_data_003['Cell type task']['title'], task_data_003['Cell type task']['rows'], source_colors_003)
axes[1, 0].set_ylabel('F1-macro', fontsize=15.0)
axes[1, 1].set_ylabel('')
axes[1, 0].set_title('')
axes[1, 1].set_title('')

# Row 3 (002)
task_data_002 = {}
for task_name, cfg in TASKS.items():
    fr = load_full_results_2(cfg, RESOURCES_002)
    rows = build_plot_data_2(fr)
    task_data_002[task_name] = {'rows': rows, 'title': cfg['title']}
source_colors_002 = {r['name']: SOURCE_BAND_COLORS[i % len(SOURCE_BAND_COLORS)] for i, r in enumerate(RESOURCES_002)}
draw_task_dataset_axis_2(axes[2, 0], task_data_002['Niche task']['title'], task_data_002['Niche task']['rows'], source_colors_002)
draw_task_dataset_axis_2(axes[2, 1], task_data_002['Cell type task']['title'], task_data_002['Cell type task']['rows'], source_colors_002)
axes[2, 0].set_ylabel('F1-macro', fontsize=15.0)
axes[2, 1].set_ylabel('')
axes[2, 0].set_title('')
axes[2, 1].set_title('')

# Shared legend from top row only
if legend_bits:
    walk_handles, model_size_handles, source_handles, base_handles = legend_bits
    legend_ax = fig.add_axes([0.77, 0.738, 0.14, 0.225])
    legend_ax.axis('off')
    section_walk = Line2D([0], [0], color='none', marker='None', linewidth=0.0, label='Walk Length')
    section_model_size = Line2D([0], [0], color='none', marker='None', linewidth=0.0, label='Model Size')
    section_source = Line2D([0], [0], color='none', marker='None', linewidth=0.0, label='Source')
    section_gap = Line2D([], [], linestyle='None', label=' ')
    combined_handles = [section_model_size] + model_size_handles + [section_gap] + [section_walk] + walk_handles + [section_gap] + [section_source] + source_handles + [section_gap] + base_handles
    combined_labels = [h.get_label() for h in combined_handles]
    combined_leg = legend_ax.legend(combined_handles, combined_labels, loc='upper left', frameon=False, ncol=1, borderaxespad=0.0, handletextpad=0.46, handlelength=1.28, labelspacing=0.18, fontsize=10.6)
    for txt in combined_leg.get_texts():
        if txt.get_text() in ('Walk Length', 'Model Size', 'Source'):
            txt.set_fontsize(11.4)

# Number-of-walks legend for n-walk-scale row
n_walk_order = tuple(int(n) for n in N_WALKS)
n_walk_colors = ORDERED_COLORS
n_walk_handles = [
    Line2D([0], [0], color=n_walk_colors[i % len(n_walk_colors)], linestyle='None', marker='o',
           markersize=6.8 * (AMPLIFIER_MIN + ((n_walk - min(n_walk_order)) / max(max(n_walk_order) - min(n_walk_order), 1e-6)) * (AMPLIFIER_MAX - AMPLIFIER_MIN)),
           markerfacecolor=n_walk_colors[i % len(n_walk_colors)],
           markeredgecolor='white', label=format_step_label(n_walk))
    for i, n_walk in enumerate(n_walk_order)
]
n_walk_legend_ax = fig.add_axes([0.77, 0.46, 0.14, 0.12])
n_walk_legend_ax.axis('off')
section_n_walk = Line2D([0], [0], color='none', marker='None', linewidth=0.0, label='Walks')
n_walk_leg = n_walk_legend_ax.legend([section_n_walk] + n_walk_handles, [section_n_walk.get_label()] + [h.get_label() for h in n_walk_handles],
                                    loc='upper left', frameon=False, ncol=1, borderaxespad=0.0, handletextpad=0.46,
                                    handlelength=1.28, labelspacing=0.18, fontsize=10.6)
for txt in n_walk_leg.get_texts():
    if txt.get_text() == 'Walks':
        txt.set_fontsize(11.4)

# Step legend for training-scale row
step_order = tuple(int(s) for s in STEPS)
step_colors = ORDERED_COLORS
step_handles = [
    Line2D([0], [0], color=step_colors[i % len(step_colors)], linestyle='None', marker='o',
           markersize=6.8 * step_amplifier(step, step_order),
           markerfacecolor=step_colors[i % len(step_colors)],
           markeredgecolor='white', label=format_step_label(step))
    for i, step in enumerate(step_order)
]
step_legend_ax = fig.add_axes([0.77, 0.129, 0.14, 0.12])
step_legend_ax.axis('off')
section_step = Line2D([0], [0], color='none', marker='None', linewidth=0.0, label='Steps')
step_leg = step_legend_ax.legend([section_step] + step_handles, [section_step.get_label()] + [h.get_label() for h in step_handles],
                                  loc='upper left', frameon=False, ncol=1, borderaxespad=0.0, handletextpad=0.46,
                                  handlelength=1.28, labelspacing=0.18, fontsize=10.6)
for txt in step_leg.get_texts():
    if txt.get_text() == 'Steps':
        txt.set_fontsize(11.4)

fig.subplots_adjust(left=0.07, right=0.82, wspace=0.1, hspace=0.28, bottom=0.06, top=0.98)

# Row-specific x-axis labels (placed after layout adjustment)
row_label_box = dict(facecolor='white', edgecolor='none', alpha=0.9, pad=2.0)
# row1_y = max(axes[0, 0].get_position().y0 - 0.035, 0.01)
# row2_y = max(axes[1, 0].get_position().y0 - 0.035, 0.01)
# row3_y = max(axes[2, 0].get_position().y0 - 0.035, 0.01)
row1_y = max(axes[0, 0].get_position().y0 - 0.02, 0.01)
row2_y = max(axes[1, 0].get_position().y0 - 0.02, 0.01)
row3_y = max(axes[2, 0].get_position().y0 - 0.02, 0.01)
fig.text(0.445, row1_y, 'Model size', ha='center', va='center', fontsize=15.0, fontweight='regular', bbox=row_label_box)
fig.text(0.445, row2_y, 'Number of walks', ha='center', va='center', fontsize=15.0, fontweight='regular', bbox=row_label_box)
fig.text(0.445, row3_y, 'Pre-training steps', ha='center', va='center', fontsize=15.0, fontweight='regular', bbox=row_label_box)
output_dir = '/fs/ess/PAS1475/yzhong/sf_project/scripts/benchmark_results/05_19_26'
fig.savefig(os.path.join(output_dir, '010_LP_combined_view_datasets_polished_07_08.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(output_dir, '010_LP_combined_view_datasets_polished_07_08.png'), dpi=600, bbox_inches='tight')

plt.show()